In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv(r"C:\Users\nhail\Downloads\cleaned_quotes_data (1).csv")
print("Dataset Columns:", df.columns)

leakage_cols = ['OrderNumber', 'ConversionDate']
df = df.drop(columns=leakage_cols, errors='ignore')

# Define features and target (target remains as string "Won" and "Lost")
X = df.drop(columns=['QuoteStatus'], errors='ignore')
y = df['QuoteStatus']

# Identify categorical columns in X and apply Label Encoding
categorical_cols = X.select_dtypes(include=['object']).columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Split data into training and testing sets (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Initialize and train Decision Tree Classifier with a limited depth to avoid overfitting
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)


# Print Accuracy (four decimal places) and Classification Report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("Classification Report:")
# We specify labels=["Lost", "Won"] to ensure the order (Lost -> 0, Won -> 1)
# 'digits=4' ensures four decimal places in the report
print(classification_report(y_test, y_pred, labels=["Lost", "Won"], digits=4))


# (Optional) Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=["Lost", "won"])
print("\nConfusion Matrix:")
print(cm)

# (Optional) Cross-Validation
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print("\nCross-Validation Accuracy: %.2f ± %.2f" % (cv_scores.mean(), cv_scores.std()))


# (Optional) Visualize Feature Importances
feat_importances = pd.Series(clf.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
feat_importances.nlargest(10).plot(kind='barh')
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


# (Optional) Print Decision Tree Rules
tree_rules = export_text(clf, feature_names=list(X.columns))
print("\nDecision Tree Rules:")
print(tree_rules)
2